# Load extracted triples into Neo4j

Reads the pruned closed-schema triples from `data/processed/kg_triples/closed_schema_pruned.jsonl` and loads them into a local Neo4j database via `langchain_neo4j.Neo4jGraph`.

Each triple becomes a `(:Entity)-[:RELATION]->(:Entity)` pattern with these relationship properties:

| property | source |
|---|---|
| `predicate` | triple predicate |
| `pair_id` | `id` from real_contradiction |
| `side` | `t` or `h` |
| `contradiction_type` | `type` from real_contradiction (negation, lexical, numeric, ...) |
| `source_text` | original sentence the triple was extracted from |

Entities are normalized by lowercasing and trimming so surface-variant spellings collapse to the same node.

In [1]:
import json
from pathlib import Path

from langchain_neo4j import Neo4jGraph
from pydantic import BaseModel, SecretStr
from pydantic_settings import BaseSettings, SettingsConfigDict
from tqdm import tqdm

In [2]:
class Settings(BaseSettings):
    model_config = SettingsConfigDict(
        env_file=".env",
        env_file_encoding="utf-8",
        extra="ignore",
    )
    neo4j_uri: str = "bolt://localhost:7687"
    neo4j_user: str = "neo4j"
    neo4j_password: SecretStr


settings = Settings()
TRIPLES_FILE = Path("data/processed/kg_triples/closed_schema_pruned.jsonl")
print(f"Neo4j URI: {settings.neo4j_uri}")
print(f"Triples file: {TRIPLES_FILE.resolve()}")

Neo4j URI: bolt://localhost:7687
Triples file: D:\AT82.05-Claim-Contradiction-Over-Knowledge-Graphs\experiments\data\processed\kg_triples\closed_schema_pruned.jsonl


In [4]:
graph = Neo4jGraph(
    url=settings.neo4j_uri,
    username=settings.neo4j_user,
    password=settings.neo4j_password.get_secret_value(),
)
print("Connected")
print(graph.query("CALL dbms.components() YIELD name, versions RETURN name, versions"))

Connected
[{'name': 'Neo4j Kernel', 'versions': ['5.26.24']}]


In [5]:
def load_jsonl(path: Path) -> list[dict]:
    with path.open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]


records = load_jsonl(TRIPLES_FILE)
total_triples = sum(len(r["t_triples"]) + len(r["h_triples"]) for r in records)
print(f"Loaded {len(records)} pairs, {total_triples} triples")

Loaded 131 pairs, 962 triples


## Clear existing graph (optional)

Run this cell if you want to start clean. Safe because the whole graph is derived from the JSONL file and can be rebuilt by rerunning the insert cell.

In [6]:
graph.query("MATCH (n) DETACH DELETE n")
print("Graph cleared")

Graph cleared


## Insert triples

One Cypher call per triple. `MERGE` on entity name and on the (predicate, pair_id, side) composite key so rerunning this cell is idempotent.

In [7]:
INSERT_CYPHER = """
MERGE (s:Entity {name: $subject})
MERGE (o:Entity {name: $object})
MERGE (s)-[r:RELATION {predicate: $predicate, pair_id: $pair_id, side: $side}]->(o)
SET r.contradiction_type = $contradiction_type,
    r.source_text = $source_text
"""


def normalize_entity(name: str | None) -> str:
    return (name or "").strip().lower()


inserted = 0
skipped = 0
for record in tqdm(records, desc="Pairs"):
    pair_id = record["id"]
    contradiction_type = record["type"]
    for side, triples_key, text_key in (("t", "t_triples", "t_text"), ("h", "h_triples", "h_text")):
        source_text = record[text_key]
        for triple in record[triples_key]:
            subject = normalize_entity(triple.get("subject"))
            predicate = triple.get("predicate")
            obj = normalize_entity(triple.get("object"))
            if not subject or not predicate or not obj:
                skipped += 1
                continue
            graph.query(
                INSERT_CYPHER,
                params={
                    "subject": subject,
                    "predicate": predicate,
                    "object": obj,
                    "pair_id": pair_id,
                    "side": side,
                    "contradiction_type": contradiction_type,
                    "source_text": source_text,
                },
            )
            inserted += 1

print(f"Inserted: {inserted}")
print(f"Skipped (missing subject/predicate/object): {skipped}")

Pairs: 100%|██████████| 131/131 [00:06<00:00, 20.19it/s]

Inserted: 962
Skipped (missing subject/predicate/object): 0


## Verify

In [11]:
counts = graph.query(
    """
    MATCH (n:Entity)
    WITH count(n) AS entities
    MATCH ()-[r:RELATION]->()
    RETURN entities, count(r) AS relations
    """
)
print("Counts:", counts)

print("\nTop predicates in the graph:")
for row in graph.query(
    """
    MATCH ()-[r:RELATION]->()
    RETURN r.predicate AS predicate, count(*) AS n
    ORDER BY n DESC LIMIT 10
    """
):
    print(f"  {row['predicate']}: {row['n']}")

print("\nSample relations:")
for row in graph.query(
    """
    MATCH (s)-[r:RELATION]->(o)
    RETURN s.name AS subject, r.predicate AS predicate, o.name AS object,
           r.pair_id AS pair_id, r.side AS side, r.contradiction_type AS type
    LIMIT 10
    """
):
    print("  ({subject})-[:{predicate}]->({object}) [pair_id={pair_id}, side={side}, type={type}]".format(**row))

Counts: [{'entities': 1001, 'relations': 961}]

Top predicates in the graph:
  in: 139
  was: 102
  is: 95
  of: 94
  said: 85
  has: 56
  described_as: 49
  for: 45
  not_allow: 36
  by: 33

Sample relations:
  (tariq aziz)-[:was]->(member) [pair_id=1, side=t, type=negation]
  (tariq aziz)-[:not_allow]->(considered) [pair_id=1, side=t, type=negation]
  (tariq aziz)-[:was]->(in saddam's inner circle) [pair_id=1, side=h, type=negation]
  (tariq aziz)-[:was]->(outside) [pair_id=2, side=t, type=lexical]
  (tariq aziz)-[:in]->(inner circle) [pair_id=2, side=h, type=lexical]
  (inner circle)-[:of]->(saddam) [pair_id=2, side=h, type=lexical]
  (tariq aziz)-[:was]->(one) [pair_id=3, side=t, type=negation]
  (tariq aziz)-[:not_allow]->(powerful figures) [pair_id=3, side=t, type=negation]
  (powerful figures)-[:in]->(saddam's regime) [pair_id=3, side=t, type=negation]
  (tariq aziz)-[:was]->(prominent) [pair_id=3, side=h, type=negation]


## Example: same-subject, same-object, different predicate across sides

A starting point for surface-level contradiction retrieval. For each pair, find entity-node patterns where t-side and h-side disagree on the predicate. This is not yet a real contradiction detector - it just surfaces candidate conflicting triples inside the graph.

In [9]:
candidates = graph.query(
    """
    MATCH (s:Entity)-[r1:RELATION]->(o:Entity)
    MATCH (s)-[r2:RELATION]->(o)
    WHERE r1.pair_id = r2.pair_id
      AND r1.side = 't' AND r2.side = 'h'
      AND r1.predicate <> r2.predicate
    RETURN r1.pair_id AS pair_id, r1.contradiction_type AS type,
           s.name AS subject, o.name AS object,
           r1.predicate AS t_predicate, r2.predicate AS h_predicate
    ORDER BY pair_id
    LIMIT 20
    """
)
print(f"Candidate conflicting triples (t vs h, same subject+object, different predicate): {len(candidates)}")
for row in candidates:
    print("  ", row)

Candidate conflicting triples (t vs h, same subject+object, different predicate): 3
   {'pair_id': '105', 'type': 'lexical', 'subject': 'zain', 'object': 'blood', 't_predicate': 'said', 'h_predicate': 'had'}
   {'pair_id': '59', 'type': 'numeric', 'subject': 'black out', 'object': 'long beach neighbourhood', 't_predicate': 'occurred_in', 'h_predicate': 'in'}
   {'pair_id': '76', 'type': 'numeric', 'subject': 'he', 'object': 'conclusion', 't_predicate': 'was', 'h_predicate': 'had'}
